In [13]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### **Paso 1: Generación de datos**

In [14]:
# Generación de los datos sintéticos
from data_generation.data_config import DATA_CONFIG
from data_generation.PanelCreditSimulator import PanelCreditSimulator
from playground.generate_simple_panel import generate_panel

In [15]:
# Paso 1: generar los datos (por ahora hacemos uno muy sintético)
panel = generate_panel()

# Caso real
# panel_simulator = PanelCreditSimulator(DATA_CONFIG)
# panel = panel_simulator.simulate_panel()

In [16]:
# Paso 1.1: análisis exploratorio de los datos
print(panel.head())
print()
print(panel.info())
print()
print(list(panel.columns))

   firm_id  t    y_1     y_2  treated  control  cohort  cohort_start
0        0  0  7.367  46.123     True    False       0           8.0
1        0  1  9.411  46.683     True    False       0           8.0
2        0  2  7.898  53.131     True    False       0           8.0
3        0  3  9.999  47.314     True    False       0           8.0
4        0  4  8.462  50.540     True    False       0           8.0

<class 'pandas.DataFrame'>
RangeIndex: 3086 entries, 0 to 3085
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   firm_id       3086 non-null   int64  
 1   t             3086 non-null   int64  
 2   y_1           3086 non-null   float64
 3   y_2           3086 non-null   float64
 4   treated       3086 non-null   bool   
 5   control       3086 non-null   bool   
 6   cohort        3086 non-null   int64  
 7   cohort_start  1898 non-null   float64
dtypes: bool(2), float64(3), int64(3)
memory usage: 150.8 KB

### **Paso 2: Split en train y test**

In [17]:
# Divisón de IDs en train y test
from splits.split_generator import SplitGenerator

In [18]:
# Paso 2: generar el split train/test. El split_generator ya está configurado
# para que al train vayan todos los tratados y un porcentaje de los nini, y al
# test vayan todos los controles y el resto de los nini
split_generator = SplitGenerator(panel, train_nini_ratio=0.5, seed=13)
train_ids, test_ids = split_generator.generate()

In [19]:
last = panel.sort_values('t').groupby('firm_id').last()
status = last[['treated', 'control', 'cohort']].reset_index()
status

,firm_id,treated,control,cohort
0,0,True,False,0
1,1,False,False,-1
2,2,True,False,1
3,3,True,False,1
4,4,False,False,-1
...,...,...,...,...
195,195,True,False,0
196,196,False,False,-1
197,197,False,False,-1
198,198,False,False,-1


In [20]:
# Paso 2.1: revisar que el split se hizo correctamente
split = split_generator.split

train = split['train']
test = split['test']

treated = train['T']
nini_train = train['NiNi']

control = test['C']
nini_test = test['NiNi']

for id in treated:
    firm = panel.groupby("firm_id").get_group(id)
    assert firm["treated"].iloc[0] == True, f"Firm {id} is not treated"
    assert firm["control"].iloc[0] == False, f"Firm {id} is control"

for id in control:
    firm = panel.groupby("firm_id").get_group(id)
    assert firm["control"].iloc[0] == True, f"Firm {id} is not control"
    assert firm["treated"].iloc[0] == False, f"Firm {id} is not treated"

for id in nini_train + nini_test:
    firm = panel.groupby("firm_id").get_group(id)
    assert firm["treated"].iloc[0] == False and firm["control"].iloc[0] == False, f"Firm {id} is not NiNi"

### **Paso 3: Conversión de datos a tensores de PyTorch**

In [22]:
from connectors.lstm import LSTMConnector

In [ ]:
lstm_connector = LSTMConnector(
    panel=panel,
    split=split_generator.split,
    feature_cols=['y_1', 'y_2']
)

train_dataset, test_dataset = lstm_connector.convert(fit_scaler=False)